In [12]:
import baltic
import statistics
import warnings
from datetime import datetime
import re
from Bio import SeqIO
import pandas as pd
import os
import random
import baltic as bt
import json
import dendropy

In [13]:
tree_95_path = "summary-95.nwk"
tree_75_path = "summary-75.nwk"
original_path = "backbone.nwk"

paths = [tree_95_path, tree_75_path, original_path]

In [16]:
for tree in paths:
    
    print(tree)

    with open(tree, 'r') as file:
        modified = file.read()

    # removing commas between segments
    modified = re.sub(r'&rea="([^"]+)"', lambda match: f'&rea="{match.group(1).replace(",", "-")}"', modified)

    # removing quotation marks around node names
    modified = re.sub(r"'(NODE_\d+)'", r'\1', modified)

    with open(f"prepped/{tree}", "w") as output_file:
        output_file.write(modified.strip())

summary-95.nwk
summary-75.nwk
backbone.nwk


In [17]:
tree_95 = bt.loadNewick("prepped/summary-95.nwk", absoluteTime= False)
tree_75 = bt.loadNewick("prepped/summary-75.nwk", absoluteTime= False)
original = bt.loadNewick("prepped/backbone.nwk", absoluteTime= False)

In [18]:
tree_list = [tree_95, tree_75, original]

In [19]:
rea_counts = {}

for name, tree in zip(["tree_95", "tree_75", "original"], tree_list):
  
    total_events = 0
    leaf_events = 0
    node_events = 0
    
    for node in tree.Objects:
        if node.traits['is_reassorted'] == 1.0:
            rea_annotation = node.traits.get('rea')
            is_uncertain = all([g_str.startswith('_') for g_str in rea_annotation.split('-')])
            total_events += 0.5 if is_uncertain else 1
        if node.is_node() and node.traits.get('is_reassorted'):
            rea_annotation = node.traits.get('rea')
            is_uncertain = all([g_str.startswith('_') for g_str in rea_annotation.split('-')])
            node_events += 0.5 if is_uncertain else 1
        elif node.is_leaf() and node.traits.get('is_reassorted'):
            rea_annotation = node.traits.get('rea')
            is_uncertain = all([g_str.startswith('_') for g_str in rea_annotation.split('-')])
            leaf_events += 0.5 if is_uncertain else 1
        

    rea_counts[name] = total_events, leaf_events, node_events

print(rea_counts)


{'tree_95': (5, 3, 2), 'tree_75': (5, 3, 2), 'original': (5, 3, 2)}
